In [1]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install shap


   ---------------------------------------- 0.0/549.1 kB ? eta -:--:--
   ---------------------------------------- 549.1/549.1 kB 7.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ------------------- -------------------- 1.3/2.7 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 8.0 MB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   - -------------------------------------- 1.8/38.1 MB 10.7 MB/s eta 0:00:04
   --- ------------------------------------ 3.4/38.1 MB 9.1 MB/s eta 0:00:04
   ---- ----------------------------------- 4.2/38.1 MB 7.1 MB/s eta 0:00:05
   ---- ----------------------------------- 4.2/38.1 MB 7.1 MB/s eta 0:00:05
   ---- ----------------------------------- 4.5/38.1 MB 4.6 MB/s eta 0:00:08
   ---- ----------------------------------- 4.7/38.1 MB 3.8 MB/s eta 0:00:09
   ----- ---------------------------------- 5.2/38.1 MB 3.6 MB/s eta 0:00:10
   ------ --------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# NAV forecasting pipeline: baseline + RandomForest + XGBoost
# Paste into a notebook cell and run.


import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
import joblib
import math
from datetime import datetime

# GPU-capable XGBoost (optional). If not installed or GPU not set, will fallback to CPU xgboost if available.
USE_XGBOOST_GPU = True  # set True if you have GPU drivers and want to use GPU for XGBoost
try:
    import xgboost as xgb
    xgb_available = True
    if USE_XGBOOST_GPU:
        # user must ensure xgboost built with GPU support; we will set tree_method later
        pass
except Exception:
    xgb_available = False

# -------------------- USER CONFIG --------------------
INPUT_CSV = "engineered_features_elss.csv"  # produced by Section 4.1
GROUP_COL = "Scheme Code"
DATE_COL = "Date"
NAV_COL = "NAV"
TARGET_COL = "nav_next"        # predicting next-period NAV
TEST_FRAC = 0.20               # fraction of last rows per scheme reserved for test
RANDOM_STATE = 42
N_JOBS = -1
CV_SPLITS = 3                  # TimeSeriesSplit folds for RandomizedSearchCV
RANDOM_SEARCH_ITERS = 25       # small for speed; increase for thorough tuning
OUT_MODEL_PATH = "nav_forecast_model.joblib"
# -----------------------------------------------------

# 1. Load data
df = pd.read_csv(INPUT_CSV)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.sort_values([GROUP_COL, DATE_COL]).reset_index(drop=True)
print("Loaded:", df.shape)

# 2. Basic sanity checks
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column {TARGET_COL} not found. Make sure Section 4.1 created it.")
if df.duplicated(subset=[GROUP_COL, DATE_COL]).any():
    print("Warning: duplicate (scheme,date) rows found. Consider deduplicating.")

# 3. Per-scheme time-aware split (last TEST_FRAC rows per scheme -> test)
def mark_holdout_rows(g, frac=TEST_FRAC):
    n = len(g)
    if n == 0:
        return pd.Series([False]*0, index=g.index)
    cutoff_idx = int(math.ceil((1-frac) * n))
    mask = [False]*n
    for i in range(cutoff_idx, n):
        mask[i] = True
    return pd.Series(mask, index=g.index)

df["_is_test"] = df.groupby(GROUP_COL).apply(lambda g: mark_holdout_rows(g, TEST_FRAC)).reset_index(level=0, drop=True)
train_df = df[~df["_is_test"]].copy()
test_df  = df[df["_is_test"]].copy()
print("Train rows:", train_df.shape[0], "Test rows:", test_df.shape[0])

# 4. Feature selection before modeling (drop obviously non-feature columns)
# Keep numeric features; drop identifiers and targets
drop_cols = [GROUP_COL, DATE_COL, TARGET_COL, "target_next_return", "target_up", "_is_test"]
# Remove any columns that might not exist in current DF
drop_cols = [c for c in drop_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in drop_cols and c != NAV_COL]  # keep NAV_COL as predictor too
# Optionally filter out high-missing or constant columns
# Remove columns with >80% missing
miss_frac = train_df[feature_cols].isnull().mean()
feature_cols = [c for c in feature_cols if miss_frac.get(c,0) <= 0.8]
# Remove near-constant columns (std ~ 0)
stds = train_df[feature_cols].std(numeric_only=True)
feature_cols = [c for c in feature_cols if (not pd.isna(stds.get(c))) and stds.get(c) > 1e-8]

print("Using feature count:", len(feature_cols))

# 5. Prepare X/y
X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COL].astype(float).copy()
X_test  = test_df[feature_cols].copy()
y_test  = test_df[TARGET_COL].astype(float).copy()

# 6. Imputation strategy: median for numeric features
imputer = SimpleImputer(strategy="median")

# Utility: evaluation function
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    # MAPE careful when y_true close to zero; compute only where abs(y_true) > tiny
    eps = 1e-6
    mape_mask = np.abs(y_true) > eps
    if mape_mask.sum() > 0:
        mape = (np.abs((y_true[mape_mask] - y_pred[mape_mask]) / y_true[mape_mask])).mean() * 100
    else:
        mape = np.nan
    # direction accuracy: how often sign of (y_next - NAV) predicted correctly.
    # But we don't have predicted NAV_prev here; we compute direction using returns with NAV_COL in test set
    # We'll compute direction using sign(y_true - NAV_current) if NAV present in test_df
    direction_acc = None
    try:
        nav_current = test_df[NAV_COL].astype(float).values
        true_dir = (y_true.values - nav_current) > 0
        pred_dir = (y_pred - nav_current) > 0
        direction_acc = (true_dir == pred_dir).mean()
    except Exception:
        direction_acc = np.nan
    r2 = r2_score(y_true, y_pred)
    print(f"{name} -> MAE: {mae:.6f} | RMSE: {rmse:.6f} | MAPE: {mape if not np.isnan(mape) else 'NA'} | R2: {r2:.4f} | DirAcc: {direction_acc if direction_acc is not None else 'NA'}")
    return {"mae":mae, "rmse":rmse, "mape":mape, "r2":r2, "dir_acc":direction_acc}

# 7. Baseline: persistence (predict next NAV = current NAV)
y_pred_naive = test_df[NAV_COL].astype(float).values
print("\nBaseline (persistence):")
_ = evaluate_model("NaivePersistence", y_test, y_pred_naive)

# 8. Baseline linear regression
# Pipeline: imputer -> scaler -> linear model
lin_pipe = Pipeline([
    ("imputer", imputer),
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])
lin_pipe.fit(X_train, y_train)
y_pred_lr = lin_pipe.predict(X_test)
print("\nLinear Regression baseline:")
_ = evaluate_model("LinearRegression", y_test, y_pred_lr)

# 9. RandomForestRegressor (with quick randomized search)
rf_pipe = Pipeline([
    ("imputer", imputer),
    ("rf", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS))
])

rf_param_dist = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [6, 8, 12, None],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4]
}

tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
rf_search = RandomizedSearchCV(
    rf_pipe, rf_param_dist, n_iter=min(RANDOM_SEARCH_ITERS, 25),
    cv=tscv, scoring="neg_mean_absolute_error", random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=1
)
print("\nRunning RandomizedSearchCV for RandomForest (this may take a while)...")
rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)
best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)
print("\nRandomForest performance:")
rf_metrics = evaluate_model("RandomForest", y_test, y_pred_rf)

# 10. XGBoost (if available)
best_xgb = None
if xgb_available:
    xgb_pipe = Pipeline([
        ("imputer", imputer),
        ("xgb", xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=1, verbosity=0))
    ])
    xgb_param_dist = {
        "xgb__n_estimators": [100, 200, 400],
        "xgb__max_depth": [4, 6, 8],
        "xgb__learning_rate": [0.01, 0.05, 0.1],
        "xgb__subsample": [0.7, 0.9, 1.0],
        "xgb__colsample_bytree": [0.6, 0.8, 1.0]
    }
    if USE_XGBOOST_GPU:
        # try to set GPU params; only effective if xgboost supports GPU
        xgb_pipe.set_params(xgb__tree_method='gpu_hist')
    xgb_search = RandomizedSearchCV(
        xgb_pipe, xgb_param_dist, n_iter=min(20, RANDOM_SEARCH_ITERS),
        cv=tscv, scoring="neg_mean_absolute_error", random_state=RANDOM_STATE, n_jobs=1, verbose=1
    )
    print("\nRunning RandomizedSearchCV for XGBoost (this may take a while)...")
    xgb_search.fit(X_train, y_train)
    best_xgb = xgb_search.best_estimator_
    print("Best XGB params:", xgb_search.best_params_)
    y_pred_xgb = best_xgb.predict(X_test)
    print("\nXGBoost performance:")
    xgb_metrics = evaluate_model("XGBoost", y_test, y_pred_xgb)
else:
    print("\nXGBoost not available in this environment; skipping XGBoost.")

# 11. Choose best model (compare MAE on test)
candidates = {"Linear": (lin_pipe, y_pred_lr)}
candidates["RandomForest"] = (best_rf, y_pred_rf)
if best_xgb is not None:
    candidates["XGBoost"] = (best_xgb, y_pred_xgb)

best_name = None
best_mae = float("inf")
for name, (model_obj, preds) in candidates.items():
    m = mean_absolute_error(y_test, preds)
    if m < best_mae:
        best_mae = m
        best_name = name
print(f"\nBest model by MAE on test: {best_name} (MAE={best_mae:.6f})")

# 12. Save the chosen model and feature list, and a small artifact file
final_model = candidates[best_name][0]
os.makedirs("models", exist_ok=True)
joblib.dump(final_model, os.path.join("models", OUT_MODEL_PATH))
joblib.dump(feature_cols, os.path.join("models", "nav_feature_list.joblib"))
print("Saved model and feature list to models/")

# 13. SHAP explanation for tree model (RandomForest or XGBoost)
try:
    import shap
    print("\nComputing SHAP values (summary)...")
    # Use tree explainer for tree models
    if best_name == "RandomForest":
        explainer = shap.TreeExplainer(final_model.named_steps['rf'])
        X_sample = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
        shap_values = explainer.shap_values(X_sample)
        shap.summary_plot(shap_values, X_sample, show=True)
    elif best_name == "XGBoost" and best_xgb is not None:
        explainer = shap.TreeExplainer(final_model.named_steps['xgb'])
        X_sample = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
        shap_values = explainer.shap_values(X_sample)
        shap.summary_plot(shap_values, X_sample, show=True)
    else:
        print("SHAP tree explainer not applicable for linear model.")
except Exception as e:
    print("SHAP not available or failed:", str(e))

# 14. Residual diagnostics (simple)
preds = candidates[best_name][1]
residuals = y_test.values - preds
print("\nResiduals summary:")
print(pd.Series(residuals).describe())

# 15. Quick backtest: rolling one-step prediction through test period (already effectively measured)
# (Optional: You can implement a true walk-forward loop to simulate live predictions)

print("\nDone. Models trained and evaluated. Inspect printed metrics and SHAP plot for interpretability.")


Loaded: (319086, 55)
Train rows: 255407 Test rows: 63679
Using feature count: 43

Baseline (persistence):
NaivePersistence -> MAE: 0.635609 | RMSE: 1.855028 | MAPE: 0.6283687918657128 | R2: 0.9999 | DirAcc: 0.4759967964321048

Linear Regression baseline:
LinearRegression -> MAE: 0.677092 | RMSE: 1.927112 | MAPE: 0.687579042490897 | R2: 0.9999 | DirAcc: 0.49245434130561094

Running RandomizedSearchCV for RandomForest (this may take a while)...
Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best RF params: {'rf__n_estimators': 300, 'rf__min_samples_split': 5, 'rf__min_samples_leaf': 2, 'rf__max_depth': None}

RandomForest performance:
RandomForest -> MAE: 0.858718 | RMSE: 3.348004 | MAPE: 0.694044611359891 | R2: 0.9998 | DirAcc: 0.5132304213319933

XGBoost not available in this environment; skipping XGBoost.

Best model by MAE on test: Linear (MAE=0.677092)
Saved model and feature list to models/
SHAP not available or failed: No module named 'shap'

Residuals summary:
count